In [3]:
import os
import glob
import json
import pandas as pd
import re
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
import warnings
import hashlib

import nltk
from nltk.corpus import stopwords

# --------------------------
# Download punkt & stopwords if needed
# --------------------------
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

from nltk.tokenize import word_tokenize, TreebankWordTokenizer

STOPWORDS = set(stopwords.words('english'))
tokenizer = TreebankWordTokenizer()

# Silence BeautifulSoup URL warning
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

# --------------------------
# Load only a small sample
# --------------------------
INPUT_DIR = './data/unstructured/news_raw/'
files = glob.glob(os.path.join(INPUT_DIR, '*.json'))

print(f"Found {len(files)} JSON files.")

all_articles = []
MAX_ARTICLES = 10000  # <<< LIMIT TO 1000
count = 0

for file in files:
    if count >= MAX_ARTICLES:
        break
    with open(file, 'r', encoding='utf-8') as f:
        articles = json.load(f)
        for article in articles:
            if count >= MAX_ARTICLES:
                break
            all_articles.append(article)
            count += 1

print(f"Loaded {len(all_articles)} raw articles (sample).")

df = pd.DataFrame(all_articles)

# --------------------------
# Text cleaner
# --------------------------
def clean_text(text):
    if pd.isnull(text):
        return ''
    text = BeautifulSoup(text, 'html.parser').get_text()
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    try:
        tokens = word_tokenize(text)
    except LookupError:
        tokens = tokenizer.tokenize(text)
    tokens = [w for w in tokens if w not in STOPWORDS]
    return ' '.join(tokens)

# --------------------------
# Clean fields
# --------------------------
df['clean_title'] = df['title'].apply(clean_text)
df['clean_body'] = df['body'].apply(clean_text)

# --------------------------
# Remove duplicates
# --------------------------
def compute_hash(text):
    return hashlib.md5(text.encode('utf-8')).hexdigest()

df['title_hash'] = df['clean_title'].apply(compute_hash)
df['body_hash'] = df['clean_body'].apply(compute_hash)

before = len(df)
df = df.drop_duplicates(subset=['title_hash', 'body_hash'])
after = len(df)
print(f"Removed {before - after} duplicates.")

# --------------------------
# Filter for crypto keywords
# --------------------------
CRYPTO_KEYWORDS = ['bitcoin', 'btc', 'ethereum', 'eth', 'crypto', 'blockchain', 'solana', 'ada']

def contains_crypto(text):
    return any(kw in text for kw in CRYPTO_KEYWORDS)

df = df[df['clean_title'].apply(contains_crypto) | df['clean_body'].apply(contains_crypto)]

print(f"Remaining after crypto filter: {len(df)} articles.")

# --------------------------
# Save
# --------------------------
OUTPUT_PATH = './data/unstructured/news_cleaned_SAMPLE.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"[✔] Saved cleaned sample to: {OUTPUT_PATH}")


Found 45 JSON files.
Loaded 10000 raw articles (sample).
Removed 27 duplicates.
Remaining after crypto filter: 8895 articles.
[✔] Saved cleaned sample to: ./data/unstructured/news_cleaned_SAMPLE.csv


In [5]:
import os

print(os.path.exists('./data/unstructured/news_cleaned_SAMPLE.csv'))  # Should be True


True


In [6]:
import pandas as pd

try:
    df = pd.read_csv('./data/unstructured/news_cleaned_SAMPLE.csv')
    print("[✔] CSV loaded successfully!")
    print(df.info())
except PermissionError:
    print("[X] Permission denied: Still locked (unexpected after reboot).")
except Exception as e:
    print(f"[X] Error loading CSV: {e}")


[✔] CSV loaded successfully!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8895 entries, 0 to 8894
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            8895 non-null   int64 
 1   guid          8895 non-null   object
 2   published_on  8895 non-null   int64 
 3   imageurl      8895 non-null   object
 4   title         8895 non-null   object
 5   url           8895 non-null   object
 6   body          8892 non-null   object
 7   tags          8423 non-null   object
 8   lang          8895 non-null   object
 9   upvotes       8895 non-null   int64 
 10  downvotes     8895 non-null   int64 
 11  categories    8895 non-null   object
 12  source_info   8895 non-null   object
 13  source        8895 non-null   object
 14  clean_title   8895 non-null   object
 15  clean_body    8892 non-null   object
 16  title_hash    8895 non-null   object
 17  body_hash     8895 non-null   object
dtypes: int64(4), object